# Safe Cruise EDA & Modeling

This notebook demonstrates the Exploratory Data Analysis (EDA) and Modeling pipeline for the Safe Cruise project. The system classifies driving behavior into Normal, Drowsy, and Aggressive based on smartphone and vehicle sensor data.

We use **XGBoost** for modeling and rely on strict trip-grouped validation to prevent data leakage.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report

import features as F
import train_full_model as T

# Explicitly loading a sample dataset file to demonstrate pandas loading logic
sample_accel = '/Users/akash/Downloads/UAH-DRIVESET-v1/D1/20151110175712-16km-D1-NORMAL1-SECONDARY/RAW_ACCELEROMETERS.txt'
df_sample = pd.read_csv(sample_accel, sep=r'\s+', header=None, names=F.ACCEL_COLS)
print("Sample Accelerometer Data:")
display(df_sample.head())

# Loading the full compiled dataset
dataset = F.load_or_build_dataset(verbose=True)
print(f"Full dataset loaded: {len(dataset)} rows")

## Feature Engineering and Static Identifiers

We drop static route/device identifiers like `Altitude`, `Roll`, and raw `Yaw` from the dataset so the model cannot "fingerprint" specific routes or phone-mount angles. We then calculate rolling windows (5s, 15s, 60s) to evaluate behavioral shifts over time.

In [ ]:
# Engineer temporal features (jerk, rolling windows) per trip
engineered = F.engineer_dataset(dataset, group_col="Trip_ID", progress=True)

# Drop rows that don't have enough history for the rolling windows
mask = F.valid_feature_mask(engineered)
engineered = engineered[mask].reset_index(drop=True)

print(f"Usable rows after dropping warm-up periods: {len(engineered)}")
display(engineered.head())

## Grouped Splitting Strategy

A random row split over time-series data creates massive data leakage because adjacent rows are highly correlated. Instead, we use `StratifiedGroupKFold` grouped by `Trip_ID`. This ensures that the model is validated on entirely unseen trips, providing a realistic estimate of its generalizability.

In [ ]:
# Set up the grouped holdout
train_idx, test_idx, train_trips, test_trips = T.grouped_holdout(engineered)

X = engineered[F.FEATURES]
y = engineered["Risk_Level"]

print(f"Training on {len(train_idx)} rows...")

# Train the model
model = T.make_model()
model.fit(X.iloc[train_idx], y.iloc[train_idx])

# Evaluate on held-out trips
y_test = y.iloc[test_idx].to_numpy()
y_hat = model.predict(X.iloc[test_idx])

print("Classification Report on Held-Out Trips:")
print(classification_report(y_test, y_hat, target_names=F.CLASS_NAMES))